# Неделя 1. Введение в RL, многорукие бандиты, MDP

Курс Reinforcement Learning. Конспект лекции.


## 1. Что такое обучение с подкреплением

**Reinforcement Learning (RL)** — раздел машинного обучения, в котором **агент** учится
принимать решения, взаимодействуя со **средой**, чтобы максимизировать суммарную награду.

Отличия от других видов обучения:

| | Supervised Learning | Unsupervised Learning | Reinforcement Learning |
|---|---|---|---|
| Данные | размеченные пары (x, y) | неразмеченные x | последовательность взаимодействий (s, a, r, s') |
| Обратная связь | правильный ответ сразу | нет обратной связи | награда, часто отложенная и зашумлённая |
| Данные i.i.d.? | да | да | нет — данные зависят от политики агента |
| Цель | минимизировать ошибку предсказания | найти структуру в данных | максимизировать суммарную награду |

Ключевая сложность RL: агент сам порождает свои данные, и его действия влияют на то,
какие данные он увидит в будущем (non-i.i.d., feedback loop).

### Agent-Environment interaction loop

```
        ┌─────────┐
   a_t  │         │
 ──────►│  Agent  │
 │      │         │
 │      └─────────┘
 │
 │  s_t, r_t
 │
 ┌─────────┐
 │         │◄──────
 │  Environment  │      a_t
 │         │
 └─────────┘
```

На каждом шаге t:

1. Агент наблюдает состояние s_t (и, возможно, награду r_t за предыдущее действие)
2. Агент выбирает действие a_t согласно своей политике
3. Среда переходит в новое состояние s_{t+1} и выдаёт награду r_{t+1}
4. Повторить

### Примеры применений

* Игры: Atari (DQN, 2013-2015), Go/Chess/Shogi (AlphaZero), StarCraft II (AlphaStar), Dota 2 (OpenAI Five)
* Робототехника: манипуляция, локомоция, sim-to-real
* Рекомендательные системы: последовательные рекомендации, учёт long-term engagement
* Управление инфраструктурой: охлаждение дата-центров, управление трафиком
* **RL для LLM**: RLHF — дообучение языковых моделей на предпочтениях людей (неделя 14)


## 2. Exploration vs Exploitation: многорукие бандиты

Прежде чем переходить к полной постановке MDP, разберём упрощённую задачу без состояний —
**многорукий бандит** (multi-armed bandit). Она изолированно демонстрирует ключевую дилемму RL:
**exploration vs exploitation**.

### Постановка задачи

* K "рук" (действий), у каждой руки k своё неизвестное распределение награды с матожиданием μ_k
* На каждом шаге t агент выбирает руку a_t ∈ {1, ..., K} и получает награду r_t ~ P(r | a_t)
* Цель — максимизировать суммарную награду за T шагов, то есть минимизировать **regret**:

$$
R_T = T \cdot \mu^* - \mathbb{E}\left[\sum_{t=1}^{T} r_t\right], \qquad \mu^* = \max_k \mu_k
$$

Агент не знает μ_k заранее — их нужно оценивать по ходу дела, одновременно стараясь
получать как можно больше награды. В этом и есть дилемма:

* **Exploitation** — выбирать руку с текущей лучшей оценкой награды
* **Exploration** — пробовать другие руки, чтобы уточнить оценки (вдруг они лучше)

### Стратегии

**ε-greedy.** С вероятностью 1-ε выбираем руку с максимальной текущей оценкой Q̂(a),
с вероятностью ε — случайную руку. Простое и рабочее решение, но exploration не учитывает
неопределённость оценки: агент одинаково часто "трогает" хорошо изученную плохую руку
и плохо изученную руку.

**UCB1 (Upper Confidence Bound).** Выбираем руку, максимизирующую верхнюю доверительную
границу:

$$
a_t = \arg\max_a \left[ \hat{Q}(a) + c \sqrt{\frac{\ln t}{N(a)}} \right]
$$

где N(a) — число выборов руки a. Идея — "оптимизм перед лицом неопределённости"
(optimism in the face of uncertainty): чем меньше мы знаем про руку, тем больше бонус
за её выбор, и бонус со временем убывает по мере накопления статистики.

**Thompson Sampling.** Байесовский подход: храним апостериорное распределение над μ_k
(например, Beta-распределение для Bernoulli-наград), на каждом шаге сэмплируем
θ_k ~ P(μ_k | данные) для каждой руки и выбираем руку с максимальным θ_k. На практике
часто работает не хуже UCB, а иногда лучше, и естественно балансирует exploration/exploitation.

Асимптотически все три стратегии (с правильными гиперпараметрами) дают regret
O(log T), что является теоретическим нижним пределом (Lai & Robbins, 1985) —
но константы и поведение на малых T сильно различаются.


In [ ]:
# Небольшая иллюстрация: во сколько раз растёт средняя награда greedy-агента
# в зависимости от того, насколько рано он "залип" на неоптимальной руке.
# Полная реализация бандит-агентов будет на семинаре.

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
true_means = [0.2, 0.5, 0.55]  # руки: плохая, средняя, лучшая

def pure_greedy_regret(n_steps=500, n_init=1):
    Q = np.zeros(len(true_means))
    N = np.zeros(len(true_means))
    regret = np.zeros(n_steps)
    # инициализация: попробовать каждую руку n_init раз
    for a in range(len(true_means)):
        for _ in range(n_init):
            r = rng.binomial(1, true_means[a])
            N[a] += 1
            Q[a] += (r - Q[a]) / N[a]
    for t in range(n_steps):
        a = np.argmax(Q)
        r = rng.binomial(1, true_means[a])
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        regret[t] = max(true_means) - true_means[a]
    return np.cumsum(regret)

for n_init in [1, 5, 20]:
    cum_regret = pure_greedy_regret(n_init=n_init)
    plt.plot(cum_regret, label=f"pure greedy, n_init={n_init}")

plt.xlabel("шаг t")
plt.ylabel("суммарный regret")
plt.title("Чисто жадная стратегия: риск застрять на неоптимальной руке")
plt.legend()
plt.show()


Видно: при малом числе начальных проб (`n_init=1`) чисто жадная стратегия имеет заметный
шанс "залипнуть" на неоптимальной руке навсегда — её regret растёт линейно по t, а не
логарифмически. Это простейшая иллюстрация того, почему **чистый exploitation без exploration
не гарантирует сходимость к оптимуму**. На семинаре реализуем ε-greedy, UCB1 и Thompson Sampling
и сравним их regret по-настоящему.


## 3. Марковский процесс принятия решений (MDP)

Бандит — MDP без состояний (или с одним состоянием). Общая постановка задачи RL —
**Markov Decision Process**, кортеж (S, A, P, R, γ):

* **S** — множество состояний
* **A** — множество действий
* **P(s' | s, a)** — вероятность перехода в s' из s при действии a
* **R(s, a, s')** (или R(s,a), или R(s)) — награда за переход
* **γ ∈ [0, 1)** — коэффициент дисконтирования

### Марковское свойство

$$
P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots, s_0) = P(s_{t+1} \mid s_t, a_t)
$$

Будущее зависит от прошлого только через текущее состояние. Это не ограничение на
реальность, а требование к тому, **как мы определяем состояние** — если нужно, в состояние
можно включить историю (например, скорость = разница последних двух положений).

### Политика, возврат, ценность

* **Политика** π(a|s) — вероятность выбрать действие a в состоянии s (детерминированная — частный случай)
* **Return** (суммарная дисконтированная награда с шага t):

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \ldots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}
$$

* **V^π(s)** — ожидаемый return при старте из s и следовании π:

$$
V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]
$$

* **Q^π(s, a)** — ожидаемый return при старте из s, выборе a, и следовании π далее:

$$
Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]
$$

Зачем γ < 1: гарантирует сходимость суммы для бесконечных горизонтов, отражает
предпочтение более ранней награды, и математически удобен (contraction mapping —
понадобится на неделе 2 для доказательства сходимости Value Iteration).

### Уравнения Беллмана (для V и Q)

Ключевая идея — **рекурсивность** return: G_t = R_{t+1} + γ G_{t+1}. Отсюда:

$$
V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V^\pi(s') \right]
$$

$$
Q^\pi(s, a) = \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma \max_{a'}{}^{\pi} Q^\pi(s', a') \right]
$$

(для Q под max/суммой в общем случае стоит действие согласно π, не обязательно max —
max появится, когда мы перейдём к **оптимальным** V*, Q* на следующей неделе).

Связь между V и Q:

$$
V^\pi(s) = \sum_a \pi(a|s) Q^\pi(s,a), \qquad Q^\pi(s,a) = \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V^\pi(s')\right]
$$

Эти уравнения — основа для **всех** методов RL, которые мы увидим дальше: Dynamic Programming
(неделя 2) решает их напрямую через итерации, TD-обучение (неделя 3) оценивает их по
семплам, deep RL (недели 5+) аппроксимирует V/Q нейросетью.


## На семинаре

* Интерфейс Gymnasium (`reset`, `step`, `action_space`, `observation_space`)
* Реализация среды бандита с нуля и агентов ε-greedy / UCB1 / Thompson Sampling
* Первое знакомство с табличной MDP-средой (FrozenLake)

## Домашнее задание

См. `../homework/homework.ipynb`.
